# Système de recommandation agricole - Modélisation - premiers pas

### Script afin de comparer les différents modèles sur les 2 fichiers

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.svm import SVR

import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Chargement des données
df_enrichi = pd.read_csv("../data/processed/yield_df_enriched_encoded.csv")
df_conso = pd.read_csv("../data/processed/yield_df_final_conso_encoded.csv")

# Colonnes
cols_num_enrichi = ['avg_temp', 'rainfall_mm', 'pesticides_tonnes','tech_trend',
                    'irrigation_impact', 'climate_instability', 'relative_tech_intensity']

cols_cat_enrichi = ['region_australia_and_new_zealand', 'region_central_asia',
                    'region_eastern_asia', 'region_eastern_europe',
                    'region_latin_america_and_the_caribbean', 'region_melanesia',
                    'region_micronesia', 'region_northern_africa',
                    'region_northern_america', 'region_northern_europe', 'region_polynesia',
                    'region_south-eastern_asia', 'region_southern_asia',
                    'region_southern_europe', 'region_sub-saharan_africa',
                    'region_western_asia', 'region_western_europe', 'item_cassava',
                    'item_maize', 'item_plantains_and_others', 'item_potatoes', 'item_rice',
                    'item_sorghum', 'item_soybean', 'item_sweet_potatoes', 'item_wheat',
                    'item_yams', 'fertilizer_used_false', 'fertilizer_used_true',
                    'irrigation_used_false', 'irrigation_used_true',
                    'weather_condition_cloudy', 'weather_condition_rainy',
                    'weather_condition_sunny', 'soil_type_chalky', 'soil_type_clay',
                    'soil_type_loam', 'soil_type_peaty', 'soil_type_sandy',
                    'soil_type_silt'
                    ]

cols_num_conso = ["avg_temp", "rainfall_mm", "pesticides_tonnes",
                  "tech_trend", "climate_instability", "relative_tech_intensity"]

cols_cat_conso = ['region_australia_and_new_zealand', 'region_central_asia',
                  'region_eastern_asia', 'region_eastern_europe',
                  'region_latin_america_and_the_caribbean', 'region_melanesia',
                  'region_micronesia', 'region_northern_africa',
                  'region_northern_america', 'region_northern_europe', 'region_polynesia',
                  'region_south-eastern_asia', 'region_southern_asia',
                  'region_southern_europe', 'region_sub-saharan_africa',
                  'region_western_asia', 'region_western_europe', 'item_cassava',
                  'item_maize', 'item_plantains_and_others', 'item_potatoes', 'item_rice',
                  'item_sorghum', 'item_soybean', 'item_sweet_potatoes', 'item_wheat',
                  'item_yams']

# Fonction d'évaluation
def evaluate_dataset(
    df,
    dataset_name,
    numeric_cols,
    categorical_cols,
    model_name,
    model
):
    target_col = "yield"

    selected_cols = numeric_cols + categorical_cols + [target_col]
    data = df[selected_cols].copy()

    X = data.drop(columns=[target_col])
    y = data[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    # Preprocessing
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", StandardScaler(), numeric_cols),
            ("cat", "passthrough", categorical_cols)
        ]
    )
    # Définition du pipeline
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    # Cross validation avec Kfold
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "mse": "neg_mean_squared_error",
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    }
    # Enregsitrement de la cross validation
    cv_results = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    # Le modèle apprend à partir des données d’entraînement
    pipeline.fit(X_train, y_train)

    # Le modèle utilise ce qu’il a appris pour prédire
    y_pred = pipeline.predict(X_test)

    cv_rmse_scores = np.sqrt(-cv_results["test_mse"])
    cv_mae_scores = -cv_results["test_mae"]
    cv_r2_scores = cv_results["test_r2"]

    # Comparer prédiction vs réalité
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    test_mae = mean_absolute_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)

    return {
        "dataset": dataset_name,
        "model": model_name,
        "n_rows": len(data),
        "n_features": X.shape[1],
        "cv_rmse_mean": cv_rmse_scores.mean(),
        "cv_rmse_std": cv_rmse_scores.std(),
        "cv_mae_mean": cv_mae_scores.mean(),
        "cv_mae_std": cv_mae_scores.std(),
        "cv_r2_mean": cv_r2_scores.mean(),
        "cv_r2_std": cv_r2_scores.std(),
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2
    }


# Définir les modèles
models = {
    "DummyRegressor": DummyRegressor(),
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(random_state=42),
    "XGBoost": xgb.XGBRegressor(random_state=42),
    "LightGBM":lgb.LGBMRegressor(verbose=-1, random_state=42)}


# Boucler sur les datasets et les modèles
all_results = []

for model_name, model in models.items():

    result_enrichi = evaluate_dataset(
        df=df_enrichi,
        dataset_name="Fichier enrichi",
        numeric_cols=cols_num_enrichi,
        categorical_cols=cols_cat_enrichi,
        model_name=model_name,
        model=model
    )
    all_results.append(result_enrichi)

    result_conso = evaluate_dataset(
        df=df_conso,
        dataset_name="Fichier consolidé",
        numeric_cols=cols_num_conso,
        categorical_cols=cols_cat_conso,
        model_name=model_name,
        model=model
    )
    all_results.append(result_conso)


# Tableau final
comparison = pd.DataFrame(all_results)

comparison = comparison[[
    "dataset", "model",
    "n_rows", "n_features",
    "cv_rmse_mean", "cv_rmse_std",
    "cv_mae_mean", "cv_mae_std",
    "cv_r2_mean", "cv_r2_std",
    "test_rmse", "test_mae", "test_r2"
]]

pd.DataFrame(comparison.round(4).sort_values(by=["model", "test_rmse"]))

,dataset,model,n_rows,n_features,cv_rmse_mean,cv_rmse_std,cv_mae_mean,cv_mae_std,cv_r2_mean,cv_r2_std,test_rmse,test_mae,test_r2
0,Fichier enrichi,DummyRegressor,29151,47,75652.6639,1271.5658,55591.4405,612.0935,-0.0007,0.0003,73864.1004,54567.1261,-0.0012
1,Fichier consolidé,DummyRegressor,29151,33,75652.6639,1271.5658,55591.4405,612.0935,-0.0007,0.0003,73864.1004,54567.1261,-0.0012
8,Fichier enrichi,LightGBM,29151,47,26499.9431,671.1922,16859.2149,259.1014,0.8771,0.0057,25907.2720,16328.3666,0.8768
9,Fichier consolidé,LightGBM,29151,33,26298.6540,751.1947,16708.1375,268.1875,0.8790,0.0061,26035.4246,16329.4829,0.8756
2,Fichier enrichi,LinearRegression,29151,47,48458.7554,1195.8693,32385.9859,606.1592,0.5894,0.0122,48168.1604,32134.6345,0.5742
3,Fichier consolidé,LinearRegression,29151,33,48452.9898,1202.5148,32395.6681,619.6688,0.5895,0.0123,48180.7848,32150.9483,0.5740
5,Fichier consolidé,RandomForest,29151,33,18127.0200,1145.5491,8505.0325,327.6338,0.9423,0.0069,17486.8506,7796.2916,0.9439
4,Fichier enrichi,RandomForest,29151,47,18502.7297,1083.8461,9128.6996,351.2340,0.9400,0.0064,17638.6699,8281.8119,0.9429
7,Fichier consolidé,XGBoost,29151,33,21165.9514,846.7274,12710.3711,254.1819,0.9216,0.0056,20362.0486,12160.6406,0.9239
6,Fichier enrichi,XGBoost,29151,47,21594.2248,1032.8386,13005.7312,264.1568,0.9183,0.0072,20784.7675,12253.4619,0.9207


- Sur ces premiers tests on ne voit pas des meilleurs résultats pour le fichier enrichi, les informations rajoutées ne semblent pas apporter quelque chose d'utile pour nos modèles.
- On voit que le modèle de RandomForest a des meilleurs résulats (sans optimisation des meilleurs paramètres) mais suivi de très près par XGBoost et LightGBM.
- N'ayant pas une distribution linéaire, on voit que la regression linéraire ne généralise pas.